In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier,VotingClassifier
from sklearn.pipeline import Pipeline as pipeline 
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error,r2_score

In [2]:
path = '/kaggle/input/isic-2024-challenge'
train_csv = os.path.join(path,'train-metadata.csv')
test_csv = os.path.join(path,'test-metadata.csv')
train = pd.read_csv(train_csv)
test = pd.read_csv(test_csv)

In [3]:
train_data = train.drop(columns=['age_approx', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5', 'iddx_full', 'lesion_id',
                                 'mel_mitotic_index', 'mel_thick_mm', 'isic_id','tbp_lv_dnn_lesion_confidence', 'tbp_tile_type',
                                 'tbp_lv_location', 'image_type', 'tbp_lv_location_simple','patient_id', 'sex',
                                 'anatom_site_general', 'attribution', 'copyright_license'])
test_data = test.drop(columns=['age_approx','tbp_tile_type', 'tbp_lv_location', 'image_type', 'tbp_lv_location_simple','patient_id',
                               'sex','anatom_site_general', 'attribution', 'copyright_license'])

In [4]:
filenames = test['isic_id']
test_data = test_data.drop(columns='isic_id')
x = train_data.drop(columns='target')
y = train_data['target']
ROS = RandomOverSampler()
RUS = RandomUnderSampler()
x_train , y_train = ROS.fit_resample(x,y)

In [5]:
preprocess = ColumnTransformer(transformers=[('num',StandardScaler(),['clin_size_long_diam_mm', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B',
       'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext',
       'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio',
       'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB',
       'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm',
       'tbp_lv_eccentricity', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence',
       'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM',
       'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt',
       'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y',
       'tbp_lv_z'])])

In [6]:
LR = LogisticRegression()
RFC = RandomForestClassifier(n_estimators=215,max_depth=3,verbose=2,random_state=42)
HGBC = HistGradientBoostingClassifier(loss='binary_crossentropy',max_iter=260,l2_regularization=0.01,verbose=2,random_state=42)
LGBMC = LGBMClassifier(n_estimators=240,verbose=2,random_state=42)
SVC_model = SVC(random_state=42,probability=False,verbose=2)
XGBC = XGBClassifier(n_estimators=135,random_state=42,verbosity=2)

In [7]:
VC = VotingClassifier(estimators=[('xgbc',XGBC),('lr',LR),('rfc',RFC),('hgbc',HGBC),('lgbmc',LGBMC)],voting='soft',verbose=2)

In [8]:
model = pipeline(steps=[('preprocessing',preprocess),('poly_features', PolynomialFeatures(degree=2)),('ensemble',VC)])

In [9]:
model.fit(x_train,y_train)

[Voting] ..................... (1 of 5) Processing xgbc, total= 4.6min
[Voting] ....................... (2 of 5) Processing lr, total=  44.7s
building tree 1 of 215
building tree 2 of 215
building tree 3 of 215
building tree 4 of 215
building tree 5 of 215
building tree 6 of 215
building tree 7 of 215
building tree 8 of 215
building tree 9 of 215
building tree 10 of 215
building tree 11 of 215
building tree 12 of 215
building tree 13 of 215
building tree 14 of 215
building tree 15 of 215
building tree 16 of 215
building tree 17 of 215
building tree 18 of 215
building tree 19 of 215
building tree 20 of 215
building tree 21 of 215
building tree 22 of 215
building tree 23 of 215
building tree 24 of 215
building tree 25 of 215
building tree 26 of 215
building tree 27 of 215
building tree 28 of 215
building tree 29 of 215
building tree 30 of 215
building tree 31 of 215
building tree 32 of 215
building tree 33 of 215
building tree 34 of 215
building tree 35 of 215
building tree 36 of 215
bui

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:  3.1min


building tree 41 of 215
building tree 42 of 215
building tree 43 of 215
building tree 44 of 215
building tree 45 of 215
building tree 46 of 215
building tree 47 of 215
building tree 48 of 215
building tree 49 of 215
building tree 50 of 215
building tree 51 of 215
building tree 52 of 215
building tree 53 of 215
building tree 54 of 215
building tree 55 of 215
building tree 56 of 215
building tree 57 of 215
building tree 58 of 215
building tree 59 of 215
building tree 60 of 215
building tree 61 of 215
building tree 62 of 215
building tree 63 of 215
building tree 64 of 215
building tree 65 of 215
building tree 66 of 215
building tree 67 of 215
building tree 68 of 215
building tree 69 of 215
building tree 70 of 215
building tree 71 of 215
building tree 72 of 215
building tree 73 of 215
building tree 74 of 215
building tree 75 of 215
building tree 76 of 215
building tree 77 of 215
building tree 78 of 215
building tree 79 of 215
building tree 80 of 215
building tree 81 of 215
building tree 82

[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed: 12.5min


building tree 162 of 215
building tree 163 of 215
building tree 164 of 215
building tree 165 of 215
building tree 166 of 215
building tree 167 of 215
building tree 168 of 215
building tree 169 of 215
building tree 170 of 215
building tree 171 of 215
building tree 172 of 215
building tree 173 of 215
building tree 174 of 215
building tree 175 of 215
building tree 176 of 215
building tree 177 of 215
building tree 178 of 215
building tree 179 of 215
building tree 180 of 215
building tree 181 of 215
building tree 182 of 215
building tree 183 of 215
building tree 184 of 215
building tree 185 of 215
building tree 186 of 215
building tree 187 of 215
building tree 188 of 215
building tree 189 of 215
building tree 190 of 215
building tree 191 of 215
building tree 192 of 215
building tree 193 of 215
building tree 194 of 215
building tree 195 of 215
building tree 196 of 215
building tree 197 of 215
building tree 198 of 215
building tree 199 of 215
building tree 200 of 215
building tree 201 of 215


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['clin_size_long_diam_mm',
                                                   'tbp_lv_A', 'tbp_lv_Aext',
                                                   'tbp_lv_B', 'tbp_lv_Bext',
                                                   'tbp_lv_C', 'tbp_lv_Cext',
                                                   'tbp_lv_H', 'tbp_lv_Hext',
                                                   'tbp_lv_L', 'tbp_lv_Lext',
                                                   'tbp_lv_areaMM2',
                                                   'tbp_lv_area_perim_ratio',
                                                   'tbp_lv_color_std_mean',
                                                   'tbp_lv_deltaA',
                                                   'tbp_lv_deltaB',
                                                   'tbp_lv_del...
                                                             random_state=42, ...)),
                                              ('lr', LogisticRegression()),
                                              ('rfc',
                                               RandomForestClassifier(max_depth=3,
                                                                      n_estimators=215,
                                                                      random_state=42,
                                                                      verbose=2)),
                                              ('hgbc',
                                               HistGradientBoostingClassifier(l2_regularization=0.01,
                                                                              loss='binary_crossentropy',
                                                                              max_iter=260,
                                                                              random_state=42,
                                                                              verbose=2)),
                                              ('lgbmc',
                                               LGBMClassifier(n_estimators=240,
                                                              random_state=42,
                                                              verbose=2))],
                                  verbose=2, voting='soft'))])

In [10]:
prediction= model.predict_proba(test_data)

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.0s


In [11]:
prediction

array([[0.94340041, 0.0565996 ],
       [0.90779847, 0.09220153],
       [0.90706327, 0.09293672]])

In [12]:
submission = pd.DataFrame(columns = ["isic_id", "target"])
submission['isic_id'] = filenames
submission['target'] = prediction[:,1]
submission

,isic_id,target
0,ISIC_0015657,0.056600
1,ISIC_0015729,0.092202
2,ISIC_0015740,0.092937


In [13]:
submission.to_csv('submission.csv', index=False)